# Loading the CMS Upsilon Data and Computing Invariant Mass

This short notebook shows how to load the CMS Open Data Upsilon-to-dimuon candidate sample and compute the dimuon invariant mass from the two muon four-vectors. It is not a solution to the project fit; it only gets the data into a useful form.

## Relativistic Kinematics

CMS provides each muon's energy and three-momentum components. In units where $c=1$, write each muon four-vector as

$$p_i^\mu = (E_i, p_{x,i}, p_{y,i}, p_{z,i}).$$

The dimuon four-vector is the sum of the two muon four-vectors:

$$P^{\mu}_{\mu\mu} = p_1^\mu + p_2^\mu.$$

The invariant mass is the Minkowski norm of this total four-vector:

$$M_{\mu\mu}^2 = P^\mu P_\mu = (E_1 + E_2)^2 - (p_{x,1}+p_{x,2})^2 - (p_{y,1}+p_{y,2})^2 - (p_{z,1}+p_{z,2})^2.$$

The dimuon transverse momentum is also often useful:

$$p_{T,\mu\mu} = \sqrt{(p_{x,1}+p_{x,2})^2 + (p_{y,1}+p_{y,2})^2}.$$

The input energies and momenta are in GeV, so the reconstructed invariant mass is in GeV.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True})

## Load the CSV

If the file is already present at `data/raw/Ymumu.csv`, this notebook uses the local copy. Otherwise it tries the direct CMS Open Data file URLs and saves the downloaded CSV locally. Do not commit downloaded data files.

In [ ]:
if Path.cwd().name == "notebooks":
    repo_root = Path.cwd().parent
else:
    repo_root = Path.cwd()

data_dir = repo_root / "data" / "raw"
data_dir.mkdir(parents=True, exist_ok=True)
data_path = data_dir / "Ymumu.csv"

candidate_urls = [
    "https://opendata.cern.ch/record/5206/files/Ymumu.csv?download=1",
    "https://opendata.cern.ch/record/cms-5206/files/Ymumu.csv?download=1",
    "http://opendata.cern.ch/record/5206/files/Ymumu.csv?download=1",
    "http://cern.ch/opendata/record/5206/files/Ymumu.csv",
]

def load_upsilon_data(path=data_path):
    if path.exists():
        return pd.read_csv(path), str(path)

    errors = []
    for url in candidate_urls:
        try:
            table = pd.read_csv(url)
            if len(table) > 1000:
                table.to_csv(path, index=False)
                return table, url
        except Exception as exc:
            errors.append(f"{url}: {type(exc).__name__}: {exc}")

    raise FileNotFoundError(
        "Could not load the CMS Upsilon CSV automatically. Download it from "
        "https://opendata.cern.ch/record/cms-5206 and save it as "
        f"{path}. Tried:\n" + "\n".join(errors)
    )

events, source = load_upsilon_data()
print(f"Loaded {len(events):,} events from {source}")
events.head()

In [ ]:
events.columns.tolist()

## Compute the Dimuon Mass

The column names use `1` and `2` for the two muons. For example, `E1`, `px1`, `py1`, `pz1` are the energy and momentum components of the first muon.

In [ ]:
required_columns = ["E1", "px1", "py1", "pz1", "E2", "px2", "py2", "pz2"]
missing_columns = [column for column in required_columns if column not in events.columns]
if missing_columns:
    raise ValueError(f"Missing required four-vector columns: {missing_columns}")

total_energy = events["E1"] + events["E2"]
total_px = events["px1"] + events["px2"]
total_py = events["py1"] + events["py2"]
total_pz = events["pz1"] + events["pz2"]

mass_squared = total_energy**2 - total_px**2 - total_py**2 - total_pz**2
events["M_recomputed"] = np.sqrt(np.clip(mass_squared, 0.0, None))
events["dimuon_pt"] = np.sqrt(total_px**2 + total_py**2)

events[["E1", "px1", "py1", "pz1", "E2", "px2", "py2", "pz2", "M_recomputed", "dimuon_pt"]].head()

In [ ]:
events["M_recomputed"].describe()

## First Sanity-Check Plot

This plot confirms that the sample is in the Upsilon mass region. The project analysis should go beyond this by choosing a model, fitting it, assessing uncertainties, and comparing models.

In [ ]:
fig, ax = plt.subplots()
ax.hist(events["M_recomputed"], bins=100, range=(8.0, 12.0), histtype="step", lw=1.8)
ax.set_xlabel("dimuon invariant mass $M_{\\mu\\mu}$ (GeV)")
ax.set_ylabel("events / bin")
ax.set_title("CMS Open Data Upsilon candidates")
fig.tight_layout()

At this point you have the key observable for the project. A clean next step is to decide on a binning, define a smooth background model, define three- and four-peak signal models, and compare them with a defensible uncertainty statement.